In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(DEVICE)

Path("../outputs/metrics").mkdir(parents=True, exist_ok=True)
Path("../outputs/models").mkdir(parents=True, exist_ok=True)

cpu


In [3]:
with open("../data/graph/train_graph.pkl", "rb") as f:
    train_graph = pickle.load(f)

with open("../data/graph/test_graph.pkl", "rb") as f:
    test_graph = pickle.load(f)

with open("../data/graph/wallet_mapping.pkl", "rb") as f:
    wallet_mapping = pickle.load(f)

num_nodes = wallet_mapping["num_nodes"]

print("Num nodes:", num_nodes)
print("Train:", train_graph["edge_features"].shape)
print("Test:", test_graph["edge_features"].shape)

Num nodes: 108582
Train: (38454, 10)
Test: (298160, 10)


In [4]:
class EdgeDataset(Dataset):
    def __init__(self, graph):
        self.edge_index = torch.tensor(
            graph["edge_index"],
            dtype=torch.long
        )

        self.edge_features = torch.tensor(
            graph["edge_features"],
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            graph["edge_labels"],
            dtype=torch.float32
        )

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        source = self.edge_index[0, idx]
        target = self.edge_index[1, idx]

        edge_feat = self.edge_features[idx]
        label = self.labels[idx]

        return source, target, edge_feat, label

In [5]:
BATCH_SIZE = 1024

train_dataset = EdgeDataset(train_graph)
test_dataset = EdgeDataset(test_graph)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [6]:
class GraphEdgeClassifier(nn.Module):
    def __init__(
        self,
        num_nodes,
        edge_feat_dim,
        embedding_dim=64,
        hidden_dim=128
    ):
        super().__init__()

        self.node_embedding = nn.Embedding(
            num_nodes,
            embedding_dim
        )

        input_dim = embedding_dim * 2 + edge_feat_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, source, target, edge_feat):
        src_emb = self.node_embedding(source)
        tgt_emb = self.node_embedding(target)

        x = torch.cat(
            [src_emb, tgt_emb, edge_feat],
            dim=1
        )

        logits = self.mlp(x).squeeze(1)

        return logits

In [7]:
edge_feat_dim = train_graph["edge_features"].shape[1]

model = GraphEdgeClassifier(
    num_nodes=num_nodes,
    edge_feat_dim=edge_feat_dim,
    embedding_dim=64,
    hidden_dim=128
).to(DEVICE)

model

GraphEdgeClassifier(
  (node_embedding): Embedding(108582, 64)
  (mlp): Sequential(
    (0): Linear(in_features=138, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [8]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        probs = torch.sigmoid(logits)

        pt = torch.where(
            targets == 1,
            probs,
            1 - probs
        )

        focal_weight = (1 - pt) ** self.gamma

        alpha_weight = torch.where(
            targets == 1,
            self.alpha,
            1 - self.alpha
        )

        loss = alpha_weight * focal_weight * bce_loss

        return loss.mean()

In [9]:
criterion = FocalLoss(
    alpha=0.75,
    gamma=2.0
)

In [10]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

In [11]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion
):
    model.train()

    total_loss = 0

    for source, target, edge_feat, label in loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()

        logits = model(
            source,
            target,
            edge_feat
        )

        loss = criterion(logits, label)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [20]:
def evaluate(model, loader, threshold=0.3):
    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for source, target, edge_feat, label in loader:
            source = source.to(DEVICE)
            target = target.to(DEVICE)
            edge_feat = edge_feat.to(DEVICE)

            logits = model(
                source,
                target,
                edge_feat
            )

            probs = torch.sigmoid(logits)

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_labels.extend(
                label.numpy()
            )

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    preds = (all_probs >= threshold).astype(int)

    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(all_labels, preds),
        "precision": precision_score(all_labels, preds, zero_division=0),
        "recall": recall_score(all_labels, preds, zero_division=0),
        "f1": f1_score(all_labels, preds, zero_division=0),
        "roc_auc": roc_auc_score(all_labels, all_probs),
        "pr_auc": average_precision_score(all_labels, all_probs),
        "confusion_matrix": confusion_matrix(all_labels, preds)
    }

    return metrics

In [22]:
EPOCHS = 20

history = []

best_f1 = 0

In [23]:
for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    test_metrics = evaluate(
        model,
        test_loader
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
        "roc_auc": test_metrics["roc_auc"],
        "pr_auc": test_metrics["pr_auc"]
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Acc: {test_metrics['accuracy']:.4f} | "
        f"Prec: {test_metrics['precision']:.4f} | "
        f"Rec: {test_metrics['recall']:.4f} | "
        f"F1: {test_metrics['f1']:.4f} | "
        f"ROC-AUC: {test_metrics['roc_auc']:.4f} | "
        f"PR-AUC: {test_metrics['pr_auc']:.4f}"
    )

    if test_metrics["f1"] > best_f1:
        best_f1 = test_metrics["f1"]

        torch.save(
            model.state_dict(),
            "../outputs/models/graph_focal_loss_best.pt"
        )

Epoch 01 | Loss: 0.0094 | Acc: 0.9214 | Prec: 0.0010 | Rec: 0.3750 | F1: 0.0020 | ROC-AUC: 0.7380 | PR-AUC: 0.0635
Epoch 02 | Loss: 0.0089 | Acc: 0.9056 | Prec: 0.0009 | Rec: 0.4062 | F1: 0.0018 | ROC-AUC: 0.7329 | PR-AUC: 0.0634
Epoch 03 | Loss: 0.0084 | Acc: 0.9000 | Prec: 0.0009 | Rec: 0.4062 | F1: 0.0017 | ROC-AUC: 0.7323 | PR-AUC: 0.0634
Epoch 04 | Loss: 0.0078 | Acc: 0.8911 | Prec: 0.0009 | Rec: 0.4531 | F1: 0.0018 | ROC-AUC: 0.7362 | PR-AUC: 0.0635
Epoch 05 | Loss: 0.0070 | Acc: 0.9073 | Prec: 0.0009 | Rec: 0.4062 | F1: 0.0019 | ROC-AUC: 0.7373 | PR-AUC: 0.0635
Epoch 06 | Loss: 0.0064 | Acc: 0.8905 | Prec: 0.0008 | Rec: 0.4062 | F1: 0.0016 | ROC-AUC: 0.7166 | PR-AUC: 0.0632
Epoch 07 | Loss: 0.0059 | Acc: 0.9011 | Prec: 0.0009 | Rec: 0.4062 | F1: 0.0018 | ROC-AUC: 0.7375 | PR-AUC: 0.0635
Epoch 08 | Loss: 0.0050 | Acc: 0.9261 | Prec: 0.0010 | Rec: 0.3594 | F1: 0.0021 | ROC-AUC: 0.7170 | PR-AUC: 0.0633
Epoch 09 | Loss: 0.0047 | Acc: 0.9366 | Prec: 0.0011 | Rec: 0.3125 | F1: 0.0021 

In [15]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    "../outputs/metrics/graph_focal_loss_history.csv",
    index=False
)

history_df.tail()

,epoch,train_loss,accuracy,precision,recall,f1,roc_auc,pr_auc
15,16,0.003464,0.992320,0.002681,0.093750,0.005213,0.711368,0.063539
16,17,0.003031,0.989606,0.001969,0.093750,0.003857,0.705759,0.063345
17,18,0.002508,0.993306,0.002575,0.078125,0.004985,0.699444,0.063426
18,19,0.002186,0.991246,0.002346,0.093750,0.004577,0.709799,0.063471
19,20,0.002168,0.992450,0.002276,0.078125,0.004423,0.690910,0.063650


In [16]:
model.load_state_dict(
    torch.load(
        "../outputs/models/graph_focal_loss_best.pt"
    )
)

final_metrics = evaluate(
    model,
    test_loader
)

print("Final Metrics:")

for k, v in final_metrics.items():
    if k != "confusion_matrix":
        print(k, ":", v)

print("\nConfusion Matrix:")
print(final_metrics["confusion_matrix"])

Final Metrics:
accuracy : 0.9996545478937483
precision : 0.0851063829787234
recall : 0.0625
f1 : 0.07207207207207207
roc_auc : 0.7280547573181122
pr_auc : 0.06338792857803677

Confusion Matrix:
[[298053     43]
 [    60      4]]


In [17]:
final_result = {
    "model": "graph_focal_loss",
    "accuracy": final_metrics["accuracy"],
    "precision": final_metrics["precision"],
    "recall": final_metrics["recall"],
    "f1": final_metrics["f1"],
    "roc_auc": final_metrics["roc_auc"],
    "pr_auc": final_metrics["pr_auc"],
    "tn": final_metrics["confusion_matrix"][0, 0],
    "fp": final_metrics["confusion_matrix"][0, 1],
    "fn": final_metrics["confusion_matrix"][1, 0],
    "tp": final_metrics["confusion_matrix"][1, 1],
}

pd.DataFrame([final_result]).to_csv(
    "../outputs/metrics/graph_focal_loss_final.csv",
    index=False
)

final_result

{'model': 'graph_focal_loss',
 'accuracy': 0.9996545478937483,
 'precision': 0.0851063829787234,
 'recall': 0.0625,
 'f1': 0.07207207207207207,
 'roc_auc': 0.7280547573181122,
 'pr_auc': 0.06338792857803677,
 'tn': np.int64(298053),
 'fp': np.int64(43),
 'fn': np.int64(60),
 'tp': np.int64(4)}